# Vision Transformers, Self-Supervised Vision and Multimodal Models

Notebook for **Chapter 12** of the *dl* study document. Run the cells top to bottom. Read the full explanation on the site: https://dadipavan.github.io/Learning/study/dl/ch12.html

Tip: change a number, predict the output, then run the cell again.

## 12.1 The Vision Transformer (ViT)

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
img = np.arange(64.).reshape(8, 8)            # an 8x8 grey "image"
P = 4                                          # patch size
patches = img.reshape(8//P, P, 8//P, P).transpose(0, 2, 1, 3).reshape(-1, P*P)
print(patches.shape)
print(patches[0]); print(patches[1])

(4, 16)
[ 0.  1.  2.  3.  8.  9. 10. 11. 16. 17. 18. 19. 24. 25. 26. 27.]
[ 4.  5.  6.  7. 12. 13. 14. 15. 20. 21. 22. 23. 28. 29. 30. 31.]


In [ ]:
H = W = 224; P = 16; C = 3; d = 768
n = (H//P)*(W//P); print("patches", n, "values per patch", P*P*C, "tokens with CLS", n+1)
print("patch-embedding params", P*P*C*d + d)
for P_ in (32, 16, 8): print("patch", P_, "tokens", (224//P_)**2, "score cells", ((224//P_)**2+1)**2)

patches 196 values per patch 768 tokens with CLS 197
patch-embedding params 590592
patch 32 tokens 49 score cells 2500
patch 16 tokens 196 score cells 38809
patch 8 tokens 784 score cells 616225


In [ ]:
rng = np.random.default_rng(0)
x = rng.standard_normal((8, 8, 3)); Wp = rng.standard_normal((4*4*3, 5))
a = x.reshape(2,4,2,4,3).transpose(0,2,1,3,4).reshape(4, -1) @ Wp     # patches then matmul
b = np.zeros((2,2,5)); Wk = Wp.reshape(4,4,3,5)
for i in range(2):
    for j in range(2):
        b[i,j] = np.tensordot(x[i*4:i*4+4, j*4:j*4+4], Wk, axes=([0,1,2],[0,1,2]))   # conv, stride 4
print("linear patch embed == strided conv:", np.allclose(a, b.reshape(4,5)))

linear patch embed == strided conv: True


## 12.2 Self-Supervised Vision: Learning Without Labels

In [ ]:
rng = np.random.default_rng(1)
ids = rng.permutation(196); keep = np.sort(ids[:49])
print("MAE keeps", len(keep), "of 196; encoder attention cost ratio", round((50/197)**2, 3))
w_t = 1.0
for step, w_s in enumerate([0.0, 0.2, 0.4]):
    w_t = 0.9*w_t + 0.1*w_s; print(step, round(w_t, 4))

MAE keeps 49 of 196; encoder attention cost ratio 0.064
0 0.9
1 0.83
2 0.787


In [ ]:
# not run here (needs torch, timm and a download)
import timm, torch
model = timm.create_model("vit_base_patch16_224", pretrained=True, num_classes=0)  # num_classes=0: return embeddings
model.eval()
with torch.no_grad():
    emb = model(torch.randn(1, 3, 224, 224))     # a random "image": real images need the model's own preprocessing
print(emb.shape)                                  # expected: torch.Size([1, 768])

## 12.3 The Contrastive Loss and CLIP

In [ ]:
import numpy as np
np.set_printoptions(precision=4, suppress=True)
def softmax(z, axis=-1):
    z = z - z.max(axis=axis, keepdims=True); e = np.exp(z); return e / e.sum(axis=axis, keepdims=True)
def norm(a): return a / np.linalg.norm(a, axis=1, keepdims=True)
I  = norm(np.array([[1., 0.], [0., 1.], [1., 1.]]))        # image embeddings (3 images)
Tx = norm(np.array([[0.9, 0.1], [0.2, 0.8], [0.6, 0.7]]))  # their 3 matching captions
S = I @ Tx.T                                               # cosine similarities
print("cosine matrix\n", S)
def clip_loss(S, tau):
    logits = S / tau; n = len(S); y = np.arange(n)
    li = -np.log(softmax(logits, 1)[y, y])                 # image -> text (rows)
    lt = -np.log(softmax(logits, 0)[y, y])                 # text -> image (columns)
    return li, lt, (li.mean() + lt.mean()) / 2
for tau in (1.0, 0.1, 0.05):
    li, lt, L = clip_loss(S, tau); print("tau", tau, "row losses", li, "col losses", lt, "loss", round(L, 4))
print("random guess loss ln(3) =", round(np.log(3), 4))
print("perfect embeddings loss", round(clip_loss(np.eye(3), 0.1)[2], 5))

cosine matrix
 [[0.9939 0.2425 0.6508]
 [0.1104 0.9701 0.7593]
 [0.7809 0.8575 0.9971]]
tau 1.0 row losses [0.7799 0.8034 0.9841] col losses [0.7982 0.8656 0.9146] loss 0.8576
tau 0.1 row losses [0.0324 0.1147 0.3095] col losses [0.1124 0.2813 0.117 ] loss 0.1612
tau 0.05 row losses [0.001  0.0146 0.0719] col losses [0.014  0.0999 0.0095] loss 0.0352
random guess loss ln(3) = 1.0986
perfect embeddings loss 9e-05


In [ ]:
rng = np.random.default_rng(0)
for N in (4, 32, 256):
    z = norm(rng.standard_normal((N, 8))); zt = norm(z + 0.5 * rng.standard_normal((N, 8)))
    print("N", N, "loss", round(clip_loss(z @ zt.T, 0.1)[2], 4), "ln N", round(np.log(N), 3))

N 4 loss 0.866 ln N 1.386
N 32 loss 1.8725 ln N 3.466
N 256 loss 4.1002 ln N 5.545


## 12.4 Zero-Shot Classification and Vision-Language Models

In [ ]:
classes = ["a photo of a cat", "a photo of a dog", "a photo of a car"]
Tc = norm(np.array([[1., 0.2], [0.8, 0.6], [0., 1.]]))
img = norm(np.array([[0.9, 0.5]]))
p = softmax(100 * (img @ Tc.T)); print("zero-shot probs", p, "->", classes[p.argmax()])

zero-shot probs [[0.0213 0.9787 0.    ]] -> a photo of a dog


In [ ]:
# not run here
from transformers import CLIPModel, CLIPProcessor
from PIL import Image
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
proc = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")
labels = ["a photo of a cat", "a photo of a dog", "a photo of a car"]
inputs = proc(text=labels, images=Image.open("pet.jpg"), return_tensors="pt", padding=True)
probs = model(**inputs).logits_per_image.softmax(dim=1)   # one probability per label

## Chapter recap

## Practice set

**1.** (easy) How many patches does a 256×256 image give with `P = 16`? How many values per patch for an RGB image? **Answer:** `16 × 16 = 256` patches; `16·16·3 = 768` values per patch.

## Prompt pack

---
Done? Do the **Practice set** at the end of the chapter, then clear your Review cards on the site.

https://dadipavan.github.io/Learning/study/dl/ch12.html